# Airline AI Assistant

## Imports:

In [1]:
import os
from openai import OpenAI
import mysql.connector
from dotenv import load_dotenv

## Establishing Database and LLM API Connections:

In [8]:
# Establishing Database Connection:

load_dotenv(override= True)
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
api_key = os.getenv('OPEN_AI_API_KEY')

# DB Connection:
try:
    db_connection = mysql.connector.connect(
    host= 'localhost',
    user= DB_USER,
    password = DB_PASSWORD,
    database= 'flight_data',
)
    cursor = db_connection.cursor(buffered= True)
    print('Successfully Connected to MySQL database')

except mysql.connector.Error as error:
    print(f'Failed to connect to MySQL database: {error}')

Successfully Connected to MySQL database


In [3]:
# Establishing LLM API Connection:
MODEL = 'gpt-4.1-mini'

openai = OpenAI()
messages = [{'role': 'user', 'content': 'Hello!!'}]

response = openai.chat.completions.create(model= MODEL,
                                          messages= messages)

print(response.choices[0].message.content)

Hello! How can I assist you today?


## Defining System Prompt:

In [4]:
system_prompt = '''
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
'''

## Defining Functions to Get and Set Ticket Prices:

In [15]:
# Function to Get Ticket Price:

def get_ticket_price(origin_city, destination_city):
    """
    Function to Get Ticket Price between two cities.
    :param origin_city: Start City
    :param destination_city: Destination City
    :return: Price of a Ticket from Origin_city to Destination_city: Price
    """

    print(f'Tool called: get_ticket_price for {origin_city} to {destination_city}')

    # Query to Get Ticket Price:
    select_query = '''
    select price from ticket_prices
    where lower(origin_city) = lower(%s)
    and lower(destination_city) = lower(%s);
    '''

    # Executing The Query:
    cursor.execute(select_query, (origin_city, destination_city))
    result = cursor.fetchone()

    #print(result)

    if result:
        return f'Price of a Ticket from {origin_city.capitalize()} to {destination_city.capitalize()}: ${result[0]}'
    else:
        return f'Unknown Ticket Price for {origin_city} to {destination_city}'

In [16]:
get_ticket_price('london', 'paris')

Tool called: get_ticket_price for london to paris


'Price of a Ticket from London to Paris: $200'

In [21]:
# Function to Set Ticket Price:

def set_ticket_price(origin_city, destination_city, new_price, passcode):
    """
    Inserts a ticket price into the database between origin_city and destination_city, updates if already exists.
    :param origin_city: Starting City
    :param destination_city: Destination City
    :param new_price: Price / New Price
    :param passcode: Passcode for Authentication
    :return:
    """

    print(f'Tool called: set_ticket_price for {origin_city} to {destination_city} at ${new_price}')

    # Authenticating User:
    if passcode != 'gelC123':
        return "Action Denied: Incorrect passcode. You do not have permission to change prices."

    # Checking query to see if record already exists:
    check_query = '''
    select id from ticket_prices
    where lower(origin_city) = lower(%s) and lower(destination_city) = lower(%s);
    '''
    cursor.execute(check_query, (origin_city, destination_city))
    result = cursor.fetchone()

    # If result exists, Updating the record:
    if result:
        update_query = '''
        update ticket_prices
        set price = %s
        where id = %s
        '''
        cursor.execute(update_query, (new_price, result[0]))
        db_connection.commit()
        return f'Successfully updated the existing route from {origin_city} to {destination_city} to ${new_price}.'

    # If result does not exist, inserting new record:
    insert_query = '''
    insert into ticket_prices (origin_city, destination_city, price)
    values (%s, %s, %s);
    '''
    cursor.execute(insert_query, (origin_city, destination_city, new_price))
    db_connection.commit()
    return f'Created a new route from {origin_city} to {destination_city} and set the price to ${new_price}.'

In [19]:
# Testing set_ticket_price:

# New Record:
set_ticket_price('Mumbai','Delhi', 150, 'gelC123')

Tool called: set_ticket_price for Mumbai to Delhi at $150


'Created a new route from Mumbai to Delhi and set the price to $150.'

In [20]:
# Updating Record:
set_ticket_price('Mumbai', 'Delhi', 170, 'gelC123')

Tool called: set_ticket_price for Mumbai to Delhi at $170


'Successfully updated the existing route from Mumbai to Delhi to $170.'

## Defining Function Descriptions for LLM:

In [23]:
# JSON Schema for get_ticket_price:

get_ticket_price_schema = {
    'name': 'get_ticket_price',
    'description': 'Get the ticket price between origin_city and destination_city',
    'parameters': {
        'type': 'object',
        'properties': {
            'origin_city': {
                'type': 'string', 'description': 'The city the customer is flying from.'
            },
            'destination_city': {
                'type': 'string', 'description': 'The city the customer is flying to.'
            }
        },
        'required': ['origin_city', 'destination_city'],
        'additionalProperties': False
    }
}

In [24]:
# JSON Schema for set_ticket_price:

set_price_schema = {
    'name': 'set_ticket_price',
    'description': 'Inserts a new ticket price into the database or updates an existing route. Requires an admin passcode.',
    'parameters': {
        'type': 'object',
        'properties': {
            'origin_city': {
                'type': 'string',
                'description': 'The city the customer is flying from.',
            },
            'destination_city': {
                'type': 'string',
                'description': 'The city the customer is flying to.',
            },
            'new_price': {
                'type': 'integer',
                'description': 'The new ticket price in whole dollars.',
            },
            'passcode': {
                'type': 'string',
                'description': 'The secret admin passcode required to authorize this database change.',
            }
        },
        'required': ['origin_city', 'destination_city', 'new_price', 'passcode'],
        'additionalProperties': False,
    }
}

## Defining Tools list for LLM:

In [25]:
tools = [
    {'type': 'function', 'function': get_ticket_price_schema},
    {'type': 'function', 'function': set_price_schema}
]